# 回帰（多目的）× 分類制約：qEHVIで候補提案

**目的**：2つの連続目的を同時に最大化しながら、3クラス分類器の許容集合 `(1, 2)` に属する確率を獲得関数に反映します。多クラス分類と順序分類の両方を、モデル作成・学習・獲得関数構築・候補提案まで実行します。

**重要**：多目的の2出力は**同じ回帰モデル**の事後分布から得ます。分類モデルは別エントリで構いません。分類側の確率積はソフトな近似であり、ハード制約や厳密な同時事後分布ではありません。

[理論](../../docs/theory/30_heterogeneous_classification_constraints.md) / [Guide](../../docs/guide/30_heterogeneous_classification_constraints.md)

## 1. 二目的の回帰データと3クラスのラベル

2目的は異なる最適点を持つため、両方を同時に最大化するとトレードオフが生じます。分類ラベルは `0=不良`、`1=要確認`、`2=良好` とします。

In [ ]:
import torch
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.multi_objective.box_decompositions.non_dominated import NondominatedPartitioning

from robotorchan.acquisition import make_constrained_qehvi_acquisition
from robotorchan.models.classification.multiclass.standard.single_task import MulticlassSingleTaskGPClassifier
from robotorchan.models.classification.ordinal.standard.single_task import OrdinalSingleTaskGPClassifier
from robotorchan.models.heterogeneous import HeterogeneousModel
from robotorchan.models.standard.single_task import SingleTaskGP
from robotorchan.optim import optimize_acqf
from robotorchan.semantics import ClassificationConstraint, ProblemSemantics, RegressionObjective

torch.manual_seed(17)
train_X = torch.linspace(0, 1, 15, dtype=torch.double).unsqueeze(-1)
y1 = 1.0 - (train_X - 0.25).square()
y2 = 1.0 - (train_X - 0.78).square()
train_Y = torch.cat([y1, y2], dim=-1)
train_C = torch.bucketize(train_X.squeeze(-1), torch.tensor([0.34, 0.67], dtype=torch.double))
regression = SingleTaskGP(train_X, train_Y)
multiclass = MulticlassSingleTaskGPClassifier(train_X, train_C, num_classes=3)
ordinal = OrdinalSingleTaskGPClassifier(train_X, train_C, num_classes=3)
print("回帰出力:", tuple(train_Y.shape), "分類ラベル:", torch.bincount(train_C, minlength=3).tolist())

## 2. 分類モデルの学習と参照点

`NondominatedPartitioning` は多目的獲得関数が改善する領域を計算するためのBoTorch標準部品です。参照点は、ここでは観測目的値より十分小さい `(0, 0)` を採用します。実際の問題では、目的のスケールと意味に合わせて選んでください。

In [ ]:
multiclass.fit(epochs=20, lr=0.03)
ordinal.fit(epochs=20, lr=0.03)
ref_point = torch.tensor([0.0, 0.0], dtype=torch.double)
partitioning = NondominatedPartitioning(ref_point=ref_point, Y=train_Y)
semantics = ProblemSemantics(
    objectives=(RegressionObjective("objective_1"), RegressionObjective("objective_2")),
    constraints=(ClassificationConstraint("quality", feasible_classes=(1, 2)),),
)
bounds = torch.tensor([[0.0], [1.0]], dtype=torch.double)

## 3. 多目的回帰 × 多クラス分類

`make_constrained_qehvi_acquisition` は回帰モデルの2出力からqEHVIを構築し、分類の許容クラス確率を掛け合わせます。次の候補をSobolで探索します。

In [ ]:
heterogeneous_multi = HeterogeneousModel(
    regression, multiclass,
    output_names=["objective_1", "objective_2", "quality"],
)
acq_multi = make_constrained_qehvi_acquisition(
    heterogeneous_multi, semantics, ref_point=ref_point,
    partitioning=partitioning,
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32]), seed=17),
)
candidate_multi, score_multi = optimize_acqf(
    acq_multi, bounds, q=1, optimizer="sobol", raw_samples=128, seed=17
)
print("多目的×多クラスの候補:", candidate_multi, "獲得関数値:", score_multi)
print("許容確率:", multiclass.predict_proba(candidate_multi)[..., [1, 2]].sum(-1))
assert candidate_multi.shape == (1, 1)

## 4. 多目的回帰 × 順序分類

回帰モデル、2つの目的、参照点、Optimizerは同じまま、分類モデルだけをOrdinal GPに交換します。順序モデルの仮定によって許容確率が変わり、提案候補も変わり得ます。

In [ ]:
heterogeneous_ordinal = HeterogeneousModel(
    regression, ordinal,
    output_names=["objective_1", "objective_2", "quality"],
)
acq_ordinal = make_constrained_qehvi_acquisition(
    heterogeneous_ordinal, semantics, ref_point=ref_point,
    partitioning=partitioning,
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32]), seed=17),
)
candidate_ordinal, score_ordinal = optimize_acqf(
    acq_ordinal, bounds, q=1, optimizer="sobol", raw_samples=128, seed=17
)
print("多目的×順序分類の候補:", candidate_ordinal, "獲得関数値:", score_ordinal)
print("許容確率:", ordinal.predict_proba(candidate_ordinal)[..., [1, 2]].sum(-1))
assert candidate_ordinal.shape == (1, 1)

## 5. 結果の解釈

このNotebookは **多目的回帰 × 多クラス分類** と **多目的回帰 × 順序分類** をそれぞれモデル構築から候補提案まで示します。qEHVIは2目的の改善を評価し、分類器の許容確率がそのスコアを重み付けします。

`q>1` では分類側の確率積に独立性の近似が入ります。`probability_threshold` を使う厳密な閾値制約は扱いません。実験を実施した後は新たな2目的値と分類ラベルでモデルを更新してください。